# Training Whisper on NeMo and Megatron-core
This notebook gives an example of how to fine-tune Whisper model on LibriSpeech train-clean-100

## Model Checkpoint Downloading and Checkpoint Conversion
Download model checkpoint from Huggingface, we download the base version from: https://huggingface.co/openai/whisper-base, and convert it from huggingface format to NeMo format

In [ ]:
# checkpoint name
hf_model_id = "openai/whisper-base"

# output folders
hf_whisper = "hf_ckpt"
nemo_whisper = "nemo_ckpt"

# downloading and converting checkpoints
!huggingface-cli download $hf_model_id --local-dir $hf_whisper
!python checkpoint_converter/whisper_hf_to_nemo.py --hf_input_path $hf_whisper --output_dir $nemo_whisper

# Dataset Downloading and Manifesting
Download LibriSpeech (train-clean-100, dev-clean, test-clean) and format it into a jsonl file

The resulting file will be in the following format for every line:
```json
{
    "audio_path": "./LibriSpeech/dev-clean/251/136532/251-136532-0011.flac", 
    "text": "now it was burned away at both sides and lay still hot along the edges on the floor of the big office room in front", 
    "task": "<|transcribe|>", 
    "lang": "<|en|>", 
    "prompt": "", 
    "timestamp": "<|notimestamps|>"
}
```
- audio_path: path to the audio file
- text: the corresponding transcription
- task: task can be <|transcribe|> or <|translate|> according to your task, LibriSpeech can only be used to train transcribe task
- lang: language in the audio file (LibriSpeech is English)
- prompt: in the original training of whisper model, prompt is to place the previous transcription of the current audio file, we leave it blank here
- timestamp: if the transcription contains the timestamp information, just pass "", otherwise pass <|notimestamps|> to turn off timestamp prediction. For LibriSpeech does not contain these information, we turn off it.

In [ ]:
# path to jsonl folder
data_dir = "./data/"

# download and extract LibriSpeech
!wget https://www.openslr.org/resources/12/test-clean.tar.gz
!wget https://www.openslr.org/resources/12/dev-clean.tar.gz
!wget https://www.openslr.org/resources/12/train-clean-100.tar.gz
!tar xvzf test-clean.tar.gz --no-same-owner
!tar xvzf dev-clean.tar.gz --no-same-owner
!tar zvxf train-clean-100.tar.gz --no-same-owner

# create jsonl files
!python librispeech_manifest.py --libri_root ./LibriSpeech --splits test-clean dev-clean train-clean-100 --output_dir $data_dir --lowercase

## Model Fine-tuning
Fine-tune the Whisper model using NeMo and Megatron-core, Megatron-core features high-performance training and model parallel mechanism, but currently, this project only support for data parallel and tensor parallel, we might continue to support for pipeline parallel in the future.

Here overrides some commonly used hyper-parameters, you can also directly modify the configuration files to setup your job. Please refers to `conf/megatron_whisper_config.yaml` and `conf/megatron_model_base_config.yaml` if you want to change some settings, the comments in these configuration files provide useful information.

In [ ]:
CONFIG_DIR=nemo_whisper
CONFIG_NAME="config"
NEMO_CKPT=f"{nemo_whisper}/model.nemo"
TOKENIZER=hf_whisper

NUM_GPUS=2
NAME="megatron_whisper_ft"    # Directory to store training results
TENSOR_PARALLEL=2             # Tensor parallel size

PRECISION='bf16'     # Support for 16, bf16, 32
MAX_STEP=50
VAL_INTERVAL=25    # Validate the model every VAL_INTERVAL steps
VAL_NUMS=5         # Number of batches to run for validation step
AMP_O2=True        # For precision in bf16, turn on megatron_amp_O2 for better efficiency

LR=1.0e-4
MICRO_BATCH=4      # Batch size feed to the model in each forward path, reduce it if you face OOM issues
GLOBAL_BATCH=8     # Real logical batch size for updating
FREEZE_ENCODER=True    # Whether to freeze the encoder. It is observed that freezing the encoder sometimes give better results than fully fine-tuning

TRAIN_FILES="\[./data/train-clean-100.jsonl\]"    # Training dataset list
TRAIN_PROBS="\[1.0\]"                             # Training dataset sampling probablilty, it should have the same length with TRAIN_FILES.
VALID_FILES="\[./data/dev-clean.jsonl\]"
TEST_FILES="\[./data/test-clean.jsonl\]"
CACHE_DIR="./cache"    # Directory to store dataset cache. If you modify the jsonl files, you should delete this folder before training

!python megatron_whisper_training.py \
    --config-path $CONFIG_DIR \
    --config-name $CONFIG_NAME \
    name=$NAME \
    restore_from_path=$NEMO_CKPT \
    trainer.devices=$NUM_GPUS \
    trainer.precision=$PRECISION \
    trainer.max_steps=$MAX_STEP \
    trainer.val_check_interval=$VAL_INTERVAL \
    trainer.limit_val_batches=$VAL_NUMS \
    model.megatron_amp_O2=$AMP_O2 \
    model.micro_batch_size=$MICRO_BATCH \
    model.global_batch_size=$GLOBAL_BATCH \
    model.tensor_model_parallel_size=$TENSOR_PARALLEL \
    model.feature_extractor.path=$TOKENIZER \
    model.freeze_encoder=$FREEZE_ENCODER \
    model.tokenizer.type=$TOKENIZER \
    model.freeze_encoder=$FREEZE_ENCODER \
    model.data.train_ds.file_names=$TRAIN_FILES \
    model.data.train_ds.index_mapping_dir=$CACHE_DIR \
    model.data.train_ds.concat_sampling_probabilities=$TRAIN_PROBS \
    model.data.validation_ds.index_mapping_dir=$CACHE_DIR \
    model.data.validation_ds.file_names=$VALID_FILES \
    model.data.test_ds.file_names=$TEST_FILES \
    model.data.test_ds.index_mapping_dir=$CACHE_DIR \
    model.optim.lr=$LR

# Model Checkpoint Conversion
The fine-tuned checkpoint will be stored in `./${NAME}/checkpoints/megatron_whisper_ft.nemo`, and we can convert it back to huggingface format for other usages.

In [ ]:
# path to store fine-tuned whisper checkpoint directory
hf_whisper_ft = "./hf_ckpt_ft/"

!python checkpoint_converter/whisper_nemo_to_hf.py \
    --hf_input_path $hf_whisper \
    --nemo_input_path $NAME/checkpoints/megatron_whisper_ft.nemo \
    --output_path $hf_whisper_ft

# Evaluation
Test the fine-tuned model works properly.

In [ ]:
import soundfile as sf
from jiwer import wer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

audio_file = "./LibriSpeech/test-clean/4970/29093/4970-29093-0006.flac"
text_gt = "law seemed to him well enough as a science but he never could discover a practical case where it appeared to him worth while to go to law and all the clients who stopped with this new clerk in the ante room of the law office where he was writing philip invariably advised to settle no matter how but settle greatly to the disgust of his employer who knew that justice between man and man could only be attained by the recognized processes with the attendant fees"
processor = WhisperProcessor.from_pretrained(hf_whisper_ft)
model = WhisperForConditionalGeneration.from_pretrained(hf_whisper_ft).cuda()

audio, sr = sf.read(audio_file)

forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task="transcribe")
input_features = processor(audio, sampling_rate=sr, return_tensors="pt").input_features.cuda()
predicted_ids = model.generate(input_features, forced_decoder_ids=forced_decoder_ids)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print('Ground Truth:', text_gt)
print('Hypothesized:', transcription)
print('Word Error Rate:', wer(text_gt, transcription))